# TP NOTÉ S8 — COMPUTER VISION CNN CHALLENGE
## 4e année Option Développeur — 3 h — Oxford-IIIT Pet Dataset (UK)

**Rendu :** un notebook Jupyter entièrement exécuté. Toute décision doit être justifiée par vos propres résultats.

**Principe d'évaluation :** démarche → code → preuves expérimentales → interprétation. Une réponse générique sans preuve issue de votre exécution ne rapporte pas les points.

## Mission — 5 pts

Construire un classifieur d'images sur **Oxford-IIIT Pet**. Vous devez comparer un CNN construit par vos soins à une approche de **transfer learning**.

Le dataset contient des images de chats et chiens de plusieurs races. Votre notebook doit être exécutable sur GPU/Colab.

In [1]:
# Votre travail expérimental ici
import os
import random
import numpy as np
import torch
import torchvision
from torchvision.datasets import OxfordIIITPet

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)
print(torch.__version__)
print(torchvision.__version__)

DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

trainval_raw = OxfordIIITPet(root=DATA_DIR, split="trainval", target_types="category", download=True, transform=None)
test_raw = OxfordIIITPet(root=DATA_DIR, split="test", target_types="category", download=True, transform=None)

CLASSES = trainval_raw.classes
N_CLASSES = len(CLASSES)
print(N_CLASSES)
print(len(trainval_raw))
print(len(test_raw))
print(CLASSES[:5])

cpu
2.14.0+cpu
0.29.0+cpu


100.0%
100.0%


37
3680
3669
['Abyssinian', 'American Bulldog', 'American Pit Bull Terrier', 'Basset Hound', 'Beagle']


In [2]:
## Mission

# L'environnement d'exécution tourne sur CPU (torch 2.14.0+cpu, torchvision
# 0.29.0+cpu, aucun GPU détecté). Les temps d'entraînement seront ajustés en
# conséquence (moins d'époques et/ou sous-échantillon lors des premières
# itérations d'architecture).

# Le dataset Oxford-IIIT Pet est téléchargé et se compose de 37 classes de
# races (ex. Abyssinian, American Bulldog, American Pit Bull Terrier, Basset
# Hound, Beagle), avec 3680 images dans le split `trainval` et 3669 images
# dans le split `test` officiel, soit 7349 images au total.

# Le split `trainval` sera lui-même redécoupé en TRAIN/VALIDATION dans la
# section suivante ; le split `test` officiel ne sera utilisé qu'une seule
# fois, à la toute fin, pour l'évaluation du modèle final retenu.

# Aucune architecture n'est imposée : la mission consiste à construire un
# CNN "from scratch" et une approche de transfer learning, à les comparer
# sur un critère fixé avant lecture des résultats (accuracy et F1-macro en
# VALIDATION), puis à démontrer par l'analyse d'erreurs et un test de
# robustesse que le modèle retenu n'est pas seulement performant sur une
# métrique globale.

## Data pipeline & split — 15 pts

Construisez un pipeline d'images reproductible. Vérifiez dimensions, classes, exemples, distribution des classes et absence de chevauchement entre splits.

Justifiez redimensionnement, normalisation, batch size et stratégie d'augmentation.

In [4]:
# Votre travail expérimental ici
import collections

labels_trainval = np.array([trainval_raw._labels[i] for i in range(len(trainval_raw))])
rng = np.random.RandomState(SEED)

train_idx, val_idx = [], []
for c in range(N_CLASSES):
    idx_c = np.where(labels_trainval == c)[0]
    rng.shuffle(idx_c)
    cut = int(round(0.8 * len(idx_c)))
    train_idx.extend(idx_c[:cut])
    val_idx.extend(idx_c[cut:])

train_idx = np.array(train_idx)
val_idx = np.array(val_idx)

print(len(train_idx))
print(len(val_idx))
print(len(set(train_idx.tolist()) & set(val_idx.tolist())))
print(len(train_idx) + len(val_idx) == len(trainval_raw))

train_counts = collections.Counter(labels_trainval[train_idx])
val_counts = collections.Counter(labels_trainval[val_idx])
print(min(train_counts.values()), max(train_counts.values()))
print(len(val_counts))

2944
736
0
True
74 80
37


In [ ]:
## Data pipeline & split

# Le split stratifié TRAIN/VALIDATION (80/20, random_state=42) donne 2944
# images en TRAIN et 736 en VALIDATION, soit 3680 au total, cohérent avec la
# taille du split `trainval` brut. Aucun chevauchement d'indices n'est
# détecté entre les deux ensembles (intersection de taille 0).

# L'effectif par classe en TRAIN varie de 74 à 80 images, un écart faible
# qui confirme que la stratification a bien préservé un équilibre correct
# entre les 37 races malgré un dataset de taille modeste (~100 images par
# race avant split). Les 37 classes sont toutes représentées en VALIDATION,
# aucune race n'a disparu du split malgré le sous-échantillonnage à 20%.

# Cet équilibre par classe est important pour la suite : il évite qu'un
# écart de performance entre races en section Error analysis ne soit
# simplement un artefact de sous-représentation dans les données
# d'entraînement.

## CNN from scratch — 20 pts

Concevez un CNN comportant convolutions, activations et pooling. Le nombre de couches n'est pas imposé.

Présentez architecture, nombre de paramètres, courbes loss/metric TRAIN/VALIDATION et diagnostic underfit/overfit.

Vous devez modifier au moins une décision d'architecture après avoir observé votre première expérience, et expliquer pourquoi.

In [ ]:
# Votre travail expérimental ici


## Transfer learning — 20 pts

Utilisez une architecture pré-entraînée pertinente (par exemple une famille ResNet/MobileNet/EfficientNet disponible dans votre framework).

Comparez extraction de features puis, si pertinent, fine-tuning partiel. Documentez précisément quelles couches sont gelées/dégelées.

In [ ]:
# Votre travail expérimental ici


## Error analysis — 20 pts

Affichez et analysez au moins **24 images mal classées**.

Construisez une matrice de confusion. Identifiez les classes/races difficiles et recherchez des motifs visuels récurrents : pose, arrière-plan, cadrage, similarité morphologique, qualité d'image, etc.

Cette section pèse autant que la modélisation.

In [ ]:
# Votre travail expérimental ici


## Robustness challenge — 10 pts

Créez au moins deux perturbations raisonnables d'images (par exemple luminosité, crop, rotation ou bruit léger) et mesurez la dégradation du modèle final.

Expliquez ce que ce test suggère sur la robustesse en production.

In [ ]:
# Votre travail expérimental ici


## Final model card — 5 pts

Rédigez une mini model card : usage prévu, données, métriques, limites, classes difficiles, coût calcul, risques et amélioration prioritaire.

In [ ]:
# Votre travail expérimental ici


## Reproductibilité — 5 pts

### Exigences de preuve
Votre note dépend de résultats **propres à votre exécution** : tableaux de métriques, graphiques, observations précises, erreurs du modèle et justification des décisions.  
Vous devez conserver `random_state = 42` lorsqu'il existe, sauf lorsqu'une question vous demande explicitement d'étudier la stabilité.

### Ce qui n'est pas accepté
- une succession d'appels scikit-learn sans analyse ;
- sélectionner un modèle à partir du jeu TEST ;
- annoncer qu'un modèle est « meilleur » sans définir le critère ;
- recopier une définition théorique à la place d'une preuve expérimentale ;
- supprimer arbitrairement des données sans quantifier l'impact.